In [1]:
%reset -f
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.logger import configure
import gymnasium as gym
from gymnasium import spaces
import imageio
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Circle
import tensorflow as tf
import datetime
import os
import math
from collections import deque

KeyboardInterrupt: 

In [ ]:
#Fundamental Classes and Functions
class Weapon():
    def __init__(self, name, stockpile, range, cooldown, phit, max_salvo_size):
        self.name = name
        self.range = range
        self.stockpile = stockpile
        self.stockpile_max = stockpile
        self.cooldown = 0
        self.cooldown_max = 0
        self.phit = phit
        self.max_salvo_size = max_salvo_size

class ADMeasures:
    def __init__(self, name, stockpile, range, cooldown):
        self.name = name
        self.stockpile = stockpile
        self.stockpile_max = stockpile
        self.range = range
        self.cooldown = 0
        self.cooldown_max = cooldown

    def can_intercept(self):
        return self.stockpile > 0 and self.cooldown == 0

    def intercept(self, salvo_size):
        intercepted = min(salvo_size, self.stockpile)
        self.stockpile -= intercepted
        self.cooldown = self.cooldown_max
        return intercepted

    def reset(self):
        self.stockpile = self.stockpile_max
        self.cooldown = 0

class Ship():
    def __init__(self, name, x, y, course = 000, speed = 20, health = 5, size_of_grid = np.array([100_000, 100_000])):
        self.name = name
        self.x = x
        self.y = y
        self.course = course #In rad
        self.speed = 0 #In knots
        self.max_speed = speed
        self.health = health
        self.max_health = health
        self.size_of_grid = size_of_grid
        self.alive = True
        self.killcount = 0
        self.weapon_list = []
        self.ad_measures_list = []
        self.last_shots = []

        # For recording rolling averages.
        self.course_history = deque(maxlen=10) 
        self.speed_history = deque(maxlen=10)   
        self.course_history.append(self.course)
        self.speed_history.append(self.speed)
    
    def move(self, action):
        if self.alive == True:        
            self.course = action[0]
            self.speed = action[1] * self.max_speed 
            self.course_history.append(self.course)
            self.speed_history.append(self.speed)
            #dx and dy in [m]
            dx = self.speed * np.cos(self.course) * 1852 / 60
            dy = self.speed * np.sin(self.course) * 1852 / 60
            self.x += dx
            self.y += dy
        
    def take_damage(self):
        self.health -= 1
        if self.health <= 0:
            self.alive = False
    
    def reset_ship(self):
        self.health = self.max_health
        self.speed = 0
        self.alive = True
        for weapon in self.weapon_list:
            weapon.cooldown_timer = 0
            weapon.stockpile = weapon.stockpile_max
        for ad_measure in self.ad_measures_list:
            ad_measure.reset()

    def get_config(self):
        return (self.x, self.y, self.course, self.health, self.speed)

    def set_config(self, config):
        (self.x, self.y, self.course, self.health, self.speed) = config

    def is_alive(self):
        return self.alive

    def add_weapon(self, weapons):
        for weapon in weapons:
            self.weapon_list.append(weapon)
    
    def add_ad_measure(self, ad_measures):
        for measure in ad_measures:
            self.ad_measures_list.append(measure)

    def get_pos(self):
        return self.x, self.y

def knots_to_meter_pr_min(knots):
    return knots * 1852 / 60

def meter_pr_min_to_knots(meter_pr_min):
    return meter_pr_min * 60 / 1852

def rad_to_deg(radians):
    return radians * 180.0 / math.pi

def deg_to_rad(degrees):
    return degrees * math.pi / 180

def course_from_north_deg(radians):
    theta_deg = math.degrees(radians)  # -180..180 or 0..360
    course = (90.0 - theta_deg) % 360
    return course


In [ ]:
# Environment Class
class NavalEnvironment(gym.Env):
    def __init__(self):
        super().__init__()
        self.agent_list = []
        self.agent_start_cfg = []
        self.opponent_list = []
        self.opponent_start_cfg = []
        self.total_steps = 0
        self.current_step = 0
        self.episode_num = 1

    def define_spaces(self, size_of_grid, max_steps_pr_episode = 10000):
        # This method is used to dynamicly define the observation and action spaces of the environment
        # based on the number of agents and opponents, as well as the number of features for each agent and opponent.

        # Define max steps per episode
        self.max_steps = max_steps_pr_episode

        # Define size of grid
        self.size_of_grid = size_of_grid

        # Setup controller for opponents movements
        self.opponent_state = []
        for opponents in self.opponent_list:
            self.opponent_state.append({
                "speed_factor": 0.0,
                "course": 0.0,
                "next_change_step": 0 
            })

        # Define dimensionality of the observation space. 
        # It will be equal to the number of agents and opponents times the number of features, 
        # which is given by the  _get_state() method.
        obs_length = len(self._get_state())
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_length,), dtype=np.float32)

        # Setup Action Spaces, using course and speed. Speed is in m/min.
        # The firing action will be a float between 0 and 1, where anything above 0.5 is considered a fire action.
        # The target selection action will be a float between 0 and 1. The float will be multiplied by the number 
        # of opponents and applied a floor function to select a target.

        # Define dimensionality of the action space
        action_space_dim_low = []
        action_space_dim_high = []
        
        for agent in self.agent_list:
            # Existing movement and weapon action space
            action_space_dim_low.extend([-math.pi, 0.0])  # Course, Speed
            action_space_dim_high.extend([math.pi, 1.0])

            for weapon in agent.weapon_list:
                action_space_dim_low.extend([0.0, 0.0])  # Firing and targeting
                action_space_dim_high.extend([1.0, 1.0])

            # Add AD measure decision space
            for ad_measure in agent.ad_measures_list:
                action_space_dim_low.extend([0.0, 0.0])  # Proportion to use, selection
                action_space_dim_high.extend([1.0, 1.0])

        self.action_space = spaces.Box(
            low=np.array(action_space_dim_low, dtype=np.float32),
            high=np.array(action_space_dim_high, dtype=np.float32),
            shape=(len(action_space_dim_low),),
            dtype=np.float32
            )

    def reset(self, seed=None, options=None, fixed_start=True):
        self.episode_num += 1
        if fixed_start == True:
            # Restore each agent and opponent to their starting configuration
            for agent, start_cfg in zip(self.agent_list, self.agent_start_cfg):
                agent.set_config(start_cfg)
                agent.alive = True
                agent.reset_ship()

            for opp, start_cfg in zip(self.opponent_list, self.opponent_start_cfg):
                opp.set_config(start_cfg)
                opp.alive = True
                opp.reset_ship()
        
        # Restore each agent and opponent to a random starting configuration
        if fixed_start == False:
            for agent in self.agent_list:
                agent.x = np.random.uniform(self.size_of_grid[0], self.size_of_grid[0])
                agent.y = np.random.uniform(-self.size_of_grid[1], self.size_of_grid[1])
                agent.alive = True
                agent.reset_ship()
        
            for opponent in self.opponent_list:
                opponent.x = np.random.uniform(-self.size_of_grid[0], -self.size_of_grid[0])
                opponent.y = np.random.uniform(-self.size_of_grid[1], self.size_of_grid[1])
                opponent.killcount = True
                opponent.reset_ship()
        
        self.total_steps += self.current_step
        self.current_step = 0 #[minutes]
        return self._get_state(), {}

    def step(self, action):
        truncated = False
        terminated = False
        self.current_step += 1

        for agent in self.agent_list:
            agent.last_shots = []  # Will store dicts: {"source": (x,y), "target": (x,y), "hit": bool}
        for opponent in self.opponent_list:
            opponent.last_shots = []  # If you want to do the same for opponents

        # Adjust weapon cooldowns
        for agent in self.agent_list:
            for weapon in agent.weapon_list:
                weapon.cooldown = max(0, weapon.cooldown - 1)
        for opponent in self.opponent_list:    
            for weapon in opponent.weapon_list:
                weapon.cooldown = max(0, weapon.cooldown - 1)

        # Move opponents
        for i, opponent in enumerate(self.opponent_list):
            if opponent.alive == True:
            # Randomize speed/course at random interval
                opp_state = self.opponent_state[i]
                if self.current_step >= opp_state["next_change_step"]:
                # Randomize speed factor in [0,1]
                    opp_state["speed_factor"] = np.random.rand()
                # Randomize course in [0, 2π)
                    opp_state["course"] = np.random.uniform(-np.pi, np.pi)
                # Next change is current step + random in [20..50]
                    interval = np.random.randint(20, 51)
                    opp_state["next_change_step"] = self.current_step + interval

                # The proposed speed_factor, course for this step
                speed_factor = opp_state["speed_factor"]
                proposed_course = opp_state["course"]
                proposed_speed = speed_factor * opponent.max_speed
                dx = proposed_speed * math.cos(proposed_course) * 1852 / 60
                dy = proposed_speed * math.sin(proposed_course) * 1852 / 60
                new_x = opponent.x + dx
                new_y = opponent.y + dy

                ## Check boundary
                min_x, max_x = -self.size_of_grid[0], self.size_of_grid[0]
                min_y, max_y = -self.size_of_grid[1], self.size_of_grid[1]

                # If out of bounds, clip and bounce
                if new_x < min_x or new_x > max_x:
                    new_x = np.clip(new_x, min_x, max_x)
                    proposed_course = math.pi - proposed_course
                    proposed_course = proposed_course % (2 * np.pi)
                    opp_state["course"] = proposed_course

                if new_y < min_y or new_y > max_y:
                    new_y = np.clip(new_y, min_y, max_y)
                    proposed_course = - proposed_course
                    proposed_course = proposed_course % (2 * np.pi)
                    opp_state["course"] = proposed_course

                # Update opponents
                opponent.x = new_x
                opponent.y = new_y
                opponent.course = proposed_course
                opponent.speed = speed_factor * opponent.max_speed

        # Calculate center of gravity for each fleet 
        # Include "tactical" weight later on 
        # (e.g a ship is valued higher if it has more weapons, more crew or higher cost of building)
        avg_agent_x = np.mean([agent.x for agent in self.agent_list])
        avg_agent_y = np.mean([agent.y for agent in self.agent_list])
        avg_opponent_x = np.mean([opponent.x for opponent in self.opponent_list])
        avg_opponent_y = np.mean([opponent.y for opponent in self.opponent_list])
        cog_score = np.linalg.norm(np.array([avg_agent_x, avg_agent_y]) - np.array([avg_opponent_x, avg_opponent_y]))

        # Negatively reward the fleets for being far away from each other
        reward = - cog_score
        
        idx = 0  # This index will parse the 'action' array

        # Clear last shots
        for agent in self.agent_list:
            agent.last_shots = []
        for opponent in self.opponent_list:
            opponent.last_shots = []

        # Now handle agents' moves and firing
        num_opponents = len(self.opponent_list)
        for agent in self.agent_list:
            if not agent.alive:
                
            # skip movement
                idx += 2
            
            # skip weapons
                num_weapons = len(agent.weapon_list)
                idx += 2 * num_weapons
            
            # skip AD measures
                num_ad = len(agent.ad_measures_list)
                idx += 2 * num_ad
                continue
        
        # Movement
            course = action[idx]
            speed_factor = action[idx+1]
            idx += 2  # consume 2 floats
            agent.move([course, speed_factor])

        # Weapons
            for w in agent.weapon_list:
                firing_val = action[idx]
                targeting_val = action[idx+1]
                idx += 2

                if (firing_val > 0.5) and (w.cooldown == 0) and (w.stockpile > 0):
                    target_idx = int(math.floor(targeting_val * num_opponents))
                    target_idx = np.clip(target_idx, 0, num_opponents - 1)
                
                    chosen_opponent = self.opponent_list[target_idx]
                    dist = np.linalg.norm([agent.x - chosen_opponent.x, agent.y - chosen_opponent.y])
                
                    if dist <= w.range:
                        salvo_size = int(firing_val * w.max_salvo_size)
                        salvo_size = min(salvo_size, w.stockpile)
                    
                        agent.last_shots.append({
                            "source": (agent.x, agent.y),
                            "target": (chosen_opponent.x, chosen_opponent.y),
                            "missiles_fired": salvo_size,
                            "intercepted": 0,
                            "missiles_hit": salvo_size,
                        })
                        w.stockpile -= salvo_size
                        w.cooldown = w.cooldown_max

        # 3) AD measure usage
            for ad_m in agent.ad_measures_list:
                proportion = action[idx]
                selection  = action[idx+1]
                idx += 2

                if selection >= 0.5 and ad_m.can_intercept():
                    pass


        # Process opponent air defense measures
        for opponent in self.opponent_list:
            if not opponent.alive:
                continue
            for shot in agent.last_shots:
                if shot["target"] == (opponent.x, opponent.y):
                    for ad_measure in opponent.ad_measures_list:
                        if ad_measure.can_intercept():
                            intercepted = ad_measure.intercept(shot["missiles_hit"])
                            shot["intercepted"] += intercepted
                            shot["missiles_hit"] -= intercepted
                            if shot["missiles_hit"] <= 0:
                                break

        # Calculate distance matrix for strike opportunities for opponents
        distance_matrix = []
        for agent in self.agent_list:
            row = []
            for opponent in self.opponent_list:
                dist = np.linalg.norm([agent.x - opponent.x, agent.y - opponent.y])
                row.append(dist)
            distance_matrix.append(row)

        distance_matrix = np.array(distance_matrix)
    
        for j, opponent in enumerate(self.opponent_list):
            if opponent.alive:
                for weapon in opponent.weapon_list:
                    # the opponent targets the closest agent.
                    if weapon.cooldown == 0 and weapon.stockpile > 0:
                        dist_list = distance_matrix[:, j]
                        i_closest = np.argmin(dist_list)
                        dist_closest = dist_list[i_closest]
                    
                        if dist_closest < weapon.range:
                            # Check if that agent is alive
                            agent_target = self.agent_list[i_closest]
                            if agent_target.alive:
                                # Random chance of hitting
                                did_hit = (np.random.rand() < weapon.phit)
                            
                                opponent.last_shots.append({
                                    "source": (opponent.x, opponent.y),
                                    "target": (agent_target.x, agent_target.y),
                                    "hit": did_hit
                                })
                            
                                if did_hit:
                                    agent_target.take_damage()
                                    reward -= 50.0
                                weapon.stockpile -= 1
                                weapon.cooldown = weapon.cooldown_max

        # Check for agent deaths
        for agent in self.agent_list:
            if agent.health <= 0:
                agent.alive = False
                reward -= 100.0
                
        # Check for opponent deaths
        for opponent in self.opponent_list:
            if opponent.health <= 0:
                opponent.alive = False
                reward += 100.0
                
        # Episode terminated if all opponents or agents are dead
        all_agents_dead = all(not agent.alive for agent in self.agent_list)
        all_opponents_dead = all(not opp.alive for opp in self.opponent_list)
        done = all_agents_dead or all_opponents_dead

        if done:
            # Optionally give a finishing bonus or do any final logging
            if all_opponents_dead:
                reward += 200.0
                terminated = True
            if all_agents_dead:
                reward -= 200.0
                terminated = True
        
        # Episode truncated if max steps reached or agent out of bounds
        for agent in self.agent_list:
            if agent.x < -self.size_of_grid[0]*1.1 or agent.x > self.size_of_grid[0]*1.1 or agent.y < -self.size_of_grid[1]*1.1 or agent.y > self.size_of_grid[1]*1.1:
                truncated = True

        if self.current_step > self.max_steps:
            truncated = True

        return self._get_state(), reward, terminated, truncated, {}

    def render(self, mode="human"):
        print(f"Step: {self.current_step}, Agent Pos: {self.agent_pos}, Opponent Pos: {self.opponent_pos}")

    def add_agent(self, agent):
        self.agent_list.append(agent)
        self.agent_start_cfg.append(agent.get_config())
    
    def add_opponent(self, opponent):
        self.opponent_list.append(opponent)
        self.opponent_start_cfg.append(opponent.get_config())

    def reset_episode_num(self):
        self.episode_num = 1

    def _get_state(self):
        return np.concatenate([[np.array([agent.x, agent.y, agent.course, agent.speed, agent.health]) for agent in self.agent_list], [np.array([opponent.x, opponent.y, opponent.course, opponent.speed, opponent.health]) for opponent in self.opponent_list]]).flatten()

IndentationError: expected an indented block after function definition on line 180 (1481044409.py, line 181)

In [ ]:
def create_gif_from_env(env, model, fps=10, steps=400, gif_name="range_keeping_test.gif"):
    frames = []
    obs, _ = env.reset()
    obs = env.unwrapped._get_state()
    env.reset_episode_num()
    reward_history = []
    
    for step_i in range(steps):
        action, _ = model.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        reward_history.append(reward)  

        # If done, reset
        if terminated or truncated:
            env.reset()

        fig = plt.figure(figsize=(16, 8))
        
        gs = fig.add_gridspec(
            nrows=3, ncols=3,
            width_ratios=[1, 3, 1],
            height_ratios=[1, 1, 1] 
        )
        gs.update(hspace=1.5) 

        # Left column subplots
        ax_left_top    = fig.add_subplot(gs[0, 0])
        ax_left_middle = fig.add_subplot(gs[1, 0])
        ax_left_bottom = fig.add_subplot(gs[2, 0])


        ax_main = fig.add_subplot(gs[:, 1])
        sg_right = gs[0:3, 2].subgridspec(
            2, 1,
            height_ratios=[1, 1],
            hspace=0.8
        )

        ax_fire   = fig.add_subplot(sg_right[0, 0])
        ax_target = fig.add_subplot(sg_right[1, 0])

        heading_length = 10000

        # Plot opponents
        for opponent in env.unwrapped.opponent_list:
            if opponent.alive:
                ax_main.plot(opponent.x, opponent.y, 'ro', label=opponent.name)

                theta_immediate = opponent.course
                x_end_immediate = opponent.x + heading_length * math.cos(theta_immediate)
                y_end_immediate = opponent.y + heading_length * math.sin(theta_immediate)

                ax_main.arrow(
                    opponent.x, opponent.y,
                    x_end_immediate - opponent.x,
                    y_end_immediate - opponent.y,
                    head_width=2000,
                    head_length=3000,
                    length_includes_head=True,
                    fc='red', 
                    ec='red',
                    label='_nolegend_'
                )

                # Example text
                ax_main.text(
                    opponent.x + 5,
                    opponent.y + 5,
                    f"HP: {opponent.health}\n"
                    f"{course_from_north_deg(opponent.course):.0f}°\n"
                    f"{meter_pr_min_to_knots(opponent.speed * 1852 / 60):.1f} kn\n",
                    color='black', fontsize=8
                )

                # Range circles
                for weapon in opponent.weapon_list:
                    circle_color = 'red' if weapon.cooldown == 0 else 'pink'
                    range_circle = Circle(
                        (opponent.x, opponent.y),
                        weapon.range,
                        color=circle_color,
                        fill=False,
                        linewidth=1,
                        linestyle='-'
                    )
                    ax_main.add_patch(range_circle)

                # Firing lines
                if hasattr(opponent, "last_shots"):
                    for shot in opponent.last_shots:
                        source = shot["source"]
                        target = shot["target"]
                        color = 'red' if shot["hit"] else 'yellow'
                        ax_main.plot(
                            [source[0], target[0]], [source[1], target[1]],
                            linestyle='--', color=color, linewidth=3
                        )
            else:
                ax_main.plot(opponent.x, opponent.y, 'x', color='gray', label=f"{opponent.name} (dead)")

        # Plot agents
        for agent in env.unwrapped.agent_list:
            if agent.alive:
                ax_main.plot(agent.x, agent.y, 'bo', label=agent.name)
                
                theta_immediate = agent.course
                x_end_immediate = agent.x + heading_length * math.cos(theta_immediate)
                y_end_immediate = agent.y + heading_length * math.sin(theta_immediate)

                # Rolling average heading
                theta_avg = np.mean(agent.course_history)
                x_end_avg = agent.x + heading_length * math.cos(theta_avg)
                y_end_avg = agent.y + heading_length * math.sin(theta_avg)

                # Immediate heading line
                ax_main.plot(
                    [agent.x, x_end_immediate],
                    [agent.y, y_end_immediate],
                    color='black',
                    linestyle='--',
                    linewidth=1,
                    label='_nolegend_'
                )

                # Rolling average heading arrow
                ax_main.arrow(
                    agent.x, agent.y,
                    x_end_avg - agent.x,
                    y_end_avg - agent.y,
                    head_width=2000,
                    head_length=3000,
                    length_includes_head=True,
                    fc='blue',
                    ec='blue',
                    label='_nolegend_'
                )

                # HP, course and speed text
                ax_main.text(
                    agent.x + 5,
                    agent.y + 5,
                    f"HP: {agent.health}\n"
                    f"{course_from_north_deg(agent.course):.0f}° "
                    f"Avg: {course_from_north_deg(np.mean(agent.course_history)):.0f}°\n"
                    f"{meter_pr_min_to_knots(agent.speed * 1852 / 60):.1f}kn "
                    f"Avg: {meter_pr_min_to_knots(np.mean(np.array(agent.speed_history) * 1852 / 60)):.1f}kn",
                    color='black', fontsize=8
                )

                # Range circles
                for weapon in agent.weapon_list:
                    circle_color = 'blue' if weapon.cooldown == 0 else 'lightblue'
                    range_circle = Circle(
                        (agent.x, agent.y),
                        weapon.range,
                        color=circle_color,
                        fill=False,
                        linewidth=1,
                        linestyle='-'
                    )
                    ax_main.add_patch(range_circle)

                # Firing lines
                if hasattr(agent, "last_shots"):
                    for shot in agent.last_shots:
                        source = shot["source"]
                        target = shot["target"]
                        color = 'green' if shot["hit"] else 'yellow'
                        ax_main.plot(
                            [source[0], target[0]], [source[1], target[1]],
                            linestyle='--', color=color, linewidth=3
                        )
            else:
                ax_main.plot(agent.x, agent.y, 'x', color='gray', label=f"{agent.name} (dead)")

        ax_main.set_xlim(-env.unwrapped.size_of_grid[0], env.unwrapped.size_of_grid[0])
        ax_main.set_ylim(-env.unwrapped.size_of_grid[1], env.unwrapped.size_of_grid[1])
        ax_main.set_title(
            f"Step: {env.unwrapped.current_step} | Ep: {env.unwrapped.episode_num}\n"
            f"Time: {env.unwrapped.current_step//60}h:{env.unwrapped.current_step%60}m"
        )
        ax_main.set_xlabel("[m]")

        ax_main.axhline(y=-env.unwrapped.size_of_grid[0], color='black', linewidth=3)
        ax_main.axhline(y=env.unwrapped.size_of_grid[0], color='black', linewidth=3)
        ax_main.axvline(x=-env.unwrapped.size_of_grid[1], color='black', linewidth=3)
        ax_main.axvline(x=env.unwrapped.size_of_grid[1], color='black', linewidth=3)

        ax_fire.cla()
        ax_fire.set_title("Firing Actions")
        ax_target.cla()
        ax_target.set_title("Target Actions")

        fire_values   = []
        fire_labels   = []
        target_values = []
        target_labels = []

        idx = 0
        for agent in env.unwrapped.agent_list:
            # Skip the first 2 movement/speed actions
            idx += 2
            # Then each weapon has 2 actions Fire and Target
            for w in agent.weapon_list:
                fire_val   = action[idx]
                target_val = action[idx + 1]
                idx += 2

                # Append to lists
                fire_values.append(fire_val)
                fire_labels.append(f"{agent.name} - {w.name}")

                target_values.append(target_val)
                target_labels.append(f"{agent.name} - {w.name}")

        # Plot firing actions on ax_fire
        bar_positions_fire = np.arange(len(fire_values))
        ax_fire.bar(bar_positions_fire, fire_values, color='red')
        ax_fire.set_xticks(bar_positions_fire)
        ax_fire.set_xticklabels(fire_labels, rotation=45, ha='right')
        ax_fire.set_ylabel("Fire Value")
        ax_fire.set_ylim(-0.05, 1.05)

        # Optionally add a reference line
        ax_fire.axhline(y=0.5, color='black', linestyle='--')
        ax_fire.text(
            0.02, 0.53, 
            "Firing threshold", 
            transform=ax_fire.transAxes,
            color='black',
            fontsize=9,
            ha='left', va='bottom'
        )

        # Plot target actions on ax_target
        bar_positions_target = np.arange(len(target_values))
        ax_target.bar(bar_positions_target, target_values, color='blue')
        ax_target.set_xticks(bar_positions_target)
        ax_target.set_xticklabels(target_labels, rotation=45, ha='right')
        ax_target.set_ylabel("Target Value")
        ax_target.set_ylim(-0.05, 1.05)
        ax_target.axhline(y=0.5, color='black', linestyle='--')
        ax_target.text(
            0.02, 0.40, 
            "Opponent 1 targeted", 
            transform=ax_target.transAxes,
            color='black',
            fontsize=9,
            ha='left', va='bottom'
        )
        ax_target.text(
            0.02, 0.53, 
            "Opponent 2 targeted", 
            transform=ax_target.transAxes,
            color='black',
            fontsize=9,
            ha='left', va='bottom'
        )

        # Weapon Stockpiles
        ax_left_top.cla()
        ax_left_top.set_title("Weapon Stockpiles")

        stock_labels = []
        stockpiles   = []
        
        for ag in env.unwrapped.agent_list:
            for w in ag.weapon_list:
                label = f"{ag.name} - {w.name}"
                stock_labels.append(label)
                stockpiles.append(w.stockpile)

        for opp in env.unwrapped.opponent_list:
            for w in opp.weapon_list:
                label = f"{opp.name} - {w.name}"
                stock_labels.append(label)
                stockpiles.append(w.stockpile)

        x_positions = np.arange(len(stock_labels))
        ax_left_top.bar(x_positions, stockpiles, color='blue')
        ax_left_top.set_xticks(x_positions)
        ax_left_top.set_xticklabels(stock_labels, rotation=45, ha='right')
        ax_left_top.set_ylabel("Stockpile")
        # Collect all stockpile_max values from both agents and opponents
        all_stockpile_maxes = []

        for ag in env.unwrapped.agent_list:
            for w in ag.weapon_list:
                all_stockpile_maxes.append(w.stockpile_max)

        for opp in env.unwrapped.opponent_list:
            for w in opp.weapon_list:
                all_stockpile_maxes.append(w.stockpile_max)

        if len(all_stockpile_maxes) > 0:
            max_stockpile = max(all_stockpile_maxes)
        else:
            max_stockpile = 0

        ax_left_top.set_ylim(0, max_stockpile + 1)

        # Weapon Cooldowns
        cd_labels = []
        cd_values = []

        for ag in env.unwrapped.agent_list:
            for w in ag.weapon_list:
                label = f"{ag.name} - {w.name}"
                cd_labels.append(label)
                cd_values.append(w.cooldown)

        for opp in env.unwrapped.opponent_list:
            for w in opp.weapon_list:
                label = f"{opp.name} - {w.name}"
                cd_labels.append(label)
                cd_values.append(w.cooldown)

        bar_positions = np.arange(len(cd_labels))
        ax_left_middle.bar(bar_positions, cd_values, color='orange')
        ax_left_middle.set_xticks(bar_positions)
        ax_left_middle.set_xticklabels(cd_labels, rotation=45, ha='right')
        ax_left_middle.set_ylabel("Cooldown")

        # 2) Determine the maximum possible cooldown
        max_cooldown = max(
            (w.cooldown_max
            for ship in (env.unwrapped.agent_list + env.unwrapped.opponent_list)
            for w in ship.weapon_list),
            default=0
        )

        ax_left_middle.set_ylim(0, max_cooldown + 1)


        # Reward Over Time
        ax_left_bottom.cla()
        ax_left_bottom.set_title("Reward Over Time")
        ax_left_bottom.plot(reward_history, color='green', label='Reward')
        ax_left_bottom.set_xlabel("Steps")
        ax_left_bottom.set_ylabel("Reward")
        ax_left_bottom.legend()

        # capture frame
        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)

        plt.close(fig)

    # Save GIF
    imageio.mimsave(gif_name, frames, fps=fps)
    print(f"GIF saved as {gif_name}")


In [ ]:
# Setting up the parameters
size_of_grid = np.array([100_000, 100_000])

agent1 = Ship(name = 'Agent 1', x=-50_000, y=10_000, course=0, speed=50, health=5, size_of_grid=size_of_grid)
agent2 = Ship(name = 'Agent 2', x=-50_000, y=-10_000, course=0, speed=50, health=5, size_of_grid=size_of_grid)

opponent1 = Ship(name = 'Opponent 1', x=+50_000, y=10_000, course=0, speed=50, health=5, size_of_grid=size_of_grid)
opponent2 = Ship(name = 'Opponent 2', x=+50_000, y=-10_000, course=0, speed=50, health=5, size_of_grid=size_of_grid)

#Weapon class: name, stockpile, range [m], cooldown [min], phit
agent1.add_weapon([
    Weapon('Artillery', 10, 10_000, 1, 0.1, 10),
    Weapon('SSM - HARPOON',   10, 50_000, 50, 0.8, 10)
])

agent1.add_ad_measure([
    ADMeasures('SAM - ESSM', stockpile=5, range=60000, cooldown=5)
])

agent2.add_weapon([
    Weapon('Artillery', 10, 10_000, 1, 0.1, 10),
    Weapon('SSM - HARPOON',   10, 50_000, 50, 0.8, 10)
])

agent2.add_ad_measure([
    ADMeasures('SAM - ESSM', stockpile=5, range=60000, cooldown=5)
])

opponent1.add_weapon([
    Weapon('Artillery', 10, 10_000, 1, 0.1, 10),
    Weapon('SSM - HARPOON - ',   10, 50_000, 50, 0.8, 10)
])

opponent1.add_ad_measure([
    ADMeasures('SAM - ESSM', stockpile=5, range=60000, cooldown=5)
])

opponent2.add_weapon([
    Weapon('Artillery', 10, 10_000, 1, 0.1, 10),
    Weapon('SSM - HARPOON',   10, 50_000, 50, 0.8, 10)
])

opponent2.add_ad_measure([
    ADMeasures('SAM - ESSM', stockpile=5, range=60000, cooldown=5)
])

env = NavalEnvironment()
env.add_agent(agent1)
env.add_opponent(opponent1)
env.add_agent(agent2)
env.add_opponent(opponent2)

env.define_spaces(size_of_grid)

print(env._get_state())

[-50000  10000      0      0      5 -50000 -10000      0      0      5
  50000  10000      0      0      5  50000 -10000      0      0      5]


In [ ]:
# Logging with tensorboard, training and gif creation.
env = Monitor(env)
logdir = "./tensorboard_logs/"
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10_000, tb_log_name='Agent_vs_Random_1v0')
#100_000 = ~2.5min
create_gif_from_env(env, model)
%load_ext tensorboard

Using cpu device
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 465       |
|    ep_rew_mean     | -3.24e+07 |
| time/              |           |
|    fps             | 2186      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 2048      |
----------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 312          |
|    ep_rew_mean          | -2.13e+07    |
| time/                   |              |
|    fps                  | 1218         |
|    iterations           | 2            |
|    time_elapsed         | 3            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 2.097513e-07 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -22.7      

c:\Users\micha\anaconda3\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.reset_episode_num to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.reset_episode_num` for environment variables or `env.get_wrapper_attr('reset_episode_num')` that will search the reminding wrappers.
  logger.warn(
C:\Users\micha\AppData\Local\Temp\ipykernel_14488\337057562.py:377: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed two minor releases later. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
C:\Users\micha\AppData\Local\Temp\ipykernel_14488\337057562.py:377: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed two minor releases later. Use buffer_rgba instead.
  frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
C:\Users\micha\AppData\Local

GIF saved as range_keeping_test.gif
